# Homework 2

First I import pyspark to enable me to work with the package.

In [0]:
import pyspark

**1. What was the average time each driver spent at the pit stop for each race? Provide also the slowest and fastest pit stop in each race.**

To carry out the above-mentioned calculations, I load the pit stops dataset. I  also inspect the dataset to get an idea of the structure of the data.

In [0]:
df_pitstops = spark.read.csv("/Volumes/gr5069/raw/f1_data/pit_stops.csv", header=True)

In [0]:
display(df_pitstops)

- read.csv enabled me to load the dataset, which is in csv format. I also included "header=True" to ensure that the first row is read as headers.
- the display function enabled me to inspect the entire datasetand get all rows and columns.

To compute the average time (in milliseconds) for each driver, I import the average (avg) function. I also convert the raceId and driverId froom string datatype to integer in order to be able to compute averages in ascending order of the raceId and driverId. It is difficult to do so when the raceId and driverId are strings.


In [0]:
from pyspark.sql.functions import avg

In [0]:
# convert raceId and driverId to int
df_pitstops = df_pitstops.withColumn("raceId", df_pitstops["raceId"].cast("int")).withColumn("driverId", df_pitstops["driverId"].cast("int"))
display(df_pitstops)

In the above code, "withColumn" enabled me to replace raceId and driverId with updated versions while "cast" enabled me to change the datatypes.

In [0]:
# Calculate the average time each driver spent at the pit stop for each race
df_avg = df_pitstops.groupBy("raceId", "driverId").agg(avg("milliseconds")).orderBy("raceId", "driverId")
display(df_avg)

In the above code, "groupBy" enables me to group the data by raceId and driverId and compute the summary statistics for each group. "avg" enables me to compute the average per group, while "orderBy" enables me to sort the final results by raceId, then driverId, so that it is easier to read.

To compute the slowest and fastest pit stop for each race, I import the min and max aggregate functions. To get the slowest and fastest pit stop for each race, I group the data by raceId then calculate the maximum pit stop duration in milliseconds for each race and rename it to "slowest". I also calculate the minimum pit stop duration for each race and rename it to "fastest".

In [0]:
from pyspark.sql.functions import min, max

In [0]:
# Compute the slowest and fastest pit stop durations for each race
max_and_min = df_pitstops.groupBy("raceId").agg(max("milliseconds").alias("slowest"), min("milliseconds").alias("fastest"))
display(max_and_min)

In the above code "groupBy" enabled me to group the data by raceId, while "max" and "min" enabled me to calculate the slowest and fastest time, respectively. "alias" enabled me to rename max("milliseconds") and min("milliseconds") to "slowest" and "fastest", respectively.

**2. Rank order by finishing position the average time spent at the pit stop in each race.**

- To compute the position number of each driver based on the average pit stop time per race, I import the row_number, desc, and Window functions. "row_number" enables me to assign ranks and "Window" to define groups for ranking without collapsing rows.
- After importing the relevant functions, I create a new column called "position", which indicates the rank of each driver for each race. I then split the data into groups or windows by each race, ensuring that the ranking happens within each race and not across all races. I then order the data in ascending order, so that the fastest driver (shortest pit stop time) is position 1, and so on.
- Note: I use df_avg for the computation and not df_pitstops as the former has the relevant aggregate calculations.
- Note: I did not account for drivers who did not finish the race.

In [0]:
# Add column called position (the position of each driver in the race) and rank the drivers by their average pit stop time
from pyspark.sql.functions import row_number
from pyspark.sql.window import Window

df_avg = df_avg.withColumn("position", row_number().over(Window.partitionBy("raceId").orderBy("avg(milliseconds)")))
display(df_avg)

- In the above code, "withColumn" enables me to create a new column called "position". 
- "Window.partitionBy("raceId")" enables me to split the data into separate windows by race and ensure that the ranking happens within each race and not across all races. 
- "orderBy" enables me to sort the drivers by average pit stop time for each race. 
- "row_number().over()" enables me to assign a rank after sorting.

**3. Insert the missing code (e.g: ALO for Alonso) for drivers based on the 'drivers' dataset. Explain your logic.**

To carry out the above-mentioned task, I load the drivers dataset. I also inspect the dataset to get an idea of the structure of the data.

In [0]:
df_drivers = spark.read.csv("/Volumes/gr5069/raw/f1_data/drivers.csv", header=True)

In [0]:
display(df_drivers)

After inspecting the data, the missing values are labelled as "\N" instead of Null. I first do some cleaning by converting "/N" in the "code" column to Null. I import functions to help me with this task and the task of replacing Null with three-letter codes. Then in keeping with convention, I replace the null values of the code column with the first three letters of the driver's surname. I also ensure that the codes are in uppercase. I then inspect the data, using display, to ensure that the code worked.

In [0]:
from pyspark.sql.functions import when, col, substring, upper

# convert "\N" to NULL
df_drivers = df_drivers.withColumn("code", when(col("code") == "\\N", None).otherwise(col("code")))

# fill missing codes
df_drivers = df_drivers.withColumn("code", when(col("code").isNull(), upper(substring(col("surname"), 1, 3))).otherwise(upper(col("code"))))

- In the code above, withColumn enabled me to create a column called "code". 
- "when(col("code") == "\\N", None)" enabled me to replace the "\N" with Null values. "when" specifies the condition that needs to be satisfied for the replacement with Null values to occur.
- "otherwise(..)" enabled me to keep existing codes for none-null values.
- "(substring(col("surname"), 1, 3))" enables me to replace the Null values of code with the first three letters of the driver's surname. and otherwise enbles me too keep the existing codes as they are.
- upper ensures that all the codes are in upper case

In [0]:
display(df_drivers)

display enables me to inspect the data to ensure that everything worked.

After filling in the missing values of the "code" column, I inspect the data too ensure there are no duplicates e.g. the surnames "Morris" and "Morrison" would both produce the code "MOR". I window or group the data by unique code and then for each code, if there are duplicates, I order the group of duplicates by driverId. After that I create a temporary column called "rn", which assigns position numbers within each potential group of dupliicates.

In [0]:
# Window by code, order by driverId
win = Window.partitionBy("code").orderBy("driverId")

# Add row number within duplicates
df_drivers = df_drivers.withColumn("rn", row_number().over(win))

- In the code above, "Window.partitionBy" enabled me to window the data by code and orderBy enabled me to order poential duplicates by driverId
- "withColumn" enabled me to create a temporary column called "rn" which stored the position number of each duplicate within each window.

Before going further to change the codes of duplicates, I inspect if there are actually duplicates. When I compute the unique values of "rn", I see that there are 16 distinct values, which means there are codes with as many as 16 duplicates. I also inspect the data for examples to confirm that there are actually duplicates, and the table confirms that there are actually duplicates those with position or "rn" greater than 1 in a single window.

In [0]:
# Unique values of "rn"
df_drivers.select("rn").distinct().show()

In [0]:
# Examples of drivers with duplicate codes
df_drivers.filter(df_drivers.rn > 1).orderBy("code").show(10)

In the code above, "filter" enabled me to filter duplicates of codes i.e. those with position ("rn") > 1. orderBy enabled me to sort by the code column, for readability. "show" enabled me to display the data.

I then append numbers to duplicate codes so that no two drivers have the same code. E.g. if there are two COR, the second would be COR1. The  first code gets to keep the original.

In [0]:
# Update code: first one keeps original, and others get number suffix

from pyspark.sql.functions import col, when, concat

# Update code: first one keeps original, and others get number suffix
df_drivers = df_drivers.withColumn("code", when(col("rn") == 1, col("code")).otherwise(concat(col("code"), col("rn"))))


# Drop helper column
df_drivers = df_drivers.drop("rn")
display(df_drivers)

In the code above, "withColumn" enabled me to update the "code" column such that duplicate codes get a number suffix. I then dropped the "rn" column as I no longer needed it.

**4. Who is the youngest and the oldest driver in each race? Create a new column called “Age”. Explain your definition of "age".**

First  I will calculate the age of each driver, which is age as of April 1st, 2026. I import the lit, months_between and round functions to help me with this exercise. I will first create a column called "age". Then for each driver I will calculate how many months old they are as of April 1st, 2026. I will then convert this age in months to years by dividing by 12, and round to the nearest whole number to get the drivers' ages. After that I will inspect the data to ensure that the code worked.

In [0]:
from pyspark.sql.functions import lit, months_between, round

# Create age column and compute the ages of the drivers
df_drivers = df_drivers.withColumn("age", round(months_between(lit("2026-04-01"), col("dob")) / 12))

- In the code above "withColumn" enables me to create the column "age". 
- lit(2026-04-01) creates a constant value column with the date "2026-04-01"
- "months_between(lit("2026-04-01"), col("dob")) / 12" computes the months between the 2026-04-01 (the reference date) and the driver's date of birth, and then converts to years by dividing by 12.
- "round" rounds the age in years to the nearest whole number.

In [0]:
display(df_drivers)

After calculating the ages, I will compute the youngest and oldest driver per race. I first have to merge df_drivers to df_pitstops as the pit stops dataframe is the one with the race data. I will first group the data by raceId and then calculate the min

In [0]:
# Merge df_pitstops and df_drivers
df_pitstops = df_pitstops.join(df_drivers, ["driverId"], "inner")
display(df_pitstops)

In the above code, "join" enabled me to merge df_drivers and df_pitstops on driverId.

In [0]:
# Compute the youngest and oldest drivers for each race
ages = df_pitstops.groupBy("raceId").agg(min("age").alias("youngest"), max("age").alias("oldest"))
display(ages)

**5. At any given race, how many podiums does each driver have? create three new columns to show - on any given race - the number of wins, the number of 2nd places, and the number of 3rd places for each driver**

I will create indicators for winning and losing to enable me to provide a count of how many times a diver has won previous races, and a count of all the times a driver has not won (but completed) a race. 

In [0]:
wind = Window.partitionBy("raceId").orderBy(col("avg(milliseconds)").asc())

df_avg = df_avg.withColumn(
    "rank",
    row_number().over(wind)
)

In [0]:
# Create Win/Loss Indicators
df_avg = df_avg.withColumn("win", when(col("rank") == 1, 1).otherwise(0)) \
.withColumn("loss", when(col("rank") != 1, 1).otherwise(0))

In [0]:
# Create cumulative window
from pyspark.sql.window import Window

win_window = Window.partitionBy("driverId").orderBy("raceId").rowsBetween(Window.unboundedPreceding, -1)

In the code above "unboundedPreceding" enables me to consider rows before the current one when calculating previous wins.

In [0]:
# Create cumulative wins and losses

from pyspark.sql.functions import sum

df_avg = df_avg.withColumn("prev_wins",
    sum("win").over(win_window)
).withColumn("prev_losses",
    sum("loss").over(win_window)
)

In [0]:
display(df_avg)

The code above enables me to create columns "prev_wins" (previous wins) and "prev_losses"  on the df_avg dataframe.

**6. Continue exploring the data by answering your own question.**

Question: what is the average number of laps per race?

Group the data by raceId then calculate the average number of laps within each race

In [0]:
# Compute the average number of laps per race
avg_laps = df_pitstops.groupBy("raceId").agg(avg("lap").alias("avg_laps"))
display(avg_laps)